In [2]:
#  load the data
import pandas as pd
df_agent = pd.read_csv("agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [4]:
# The judge instructions:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [ ]:
import os

In [ ]:
from openai import OpenAI

# Resolve API key from environment if available (OPENAI_API_KEY preferred, fallback to GROQ_API_KEY)
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or GROQ_API_KEY environment variable before creating OpenAI client.")
openai_client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1") or os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")  
)

In [ ]:
# Define the judge function:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [23]:
# Test it on one agent result:
agent_eval, usage = evaluate_agent_answer(agent_answers[0], model="openai/gpt-oss-20b")

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer correctly states that a certificate cannot be earned in self‑paced mode and that a certificate requires completion of the live cohort, citing the need for participation in the capstone peer‑review process. This matches the key information from the ground truth.', answer_score='good', trajectory_reasoning='The agent made a single, relevant search query containing key terms from the question and did not issue duplicate or unnecessary calls. The number of calls was reasonable and the search could have provided supporting evidence for the answer. The tool usage was appropriate.', trajectory_score='good')

#### Running the agent judge

In [24]:
# Run the judge for all agent answers:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec, model="openai/gpt-oss-20b")

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [25]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [27]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers[:5], judge_agent_record)

  0%|          | 0/5 [00:00<?, ?it/s]

el: {'question': 'Can I take this course at my own pace and still receive a certificate at the end?', 'answer_agent': 'No — for this course, you can follow it in self-paced mode, but you **won’t receive a certificate** that way.\n\nTo get a certificate, you need to finish the course with the **live cohort**, since the certificate depends on participating in the capstone peer-review process, which is only available while the course is running.\n\nIf you want, I can also explain the certificate requirements or how the capstone/peer-review process works.', 'answer_orig': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.', 'tool_calls': '[{"name": "search", "arguments": "{\\"query\\":\\"

In [28]:
# Split the results:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [30]:
# Create a dataframe:
df_agent_eval = pd.DataFrame(agent_evaluations)
df_agent_eval

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,Can I take this course at my own pace and stil...,69d122f12e,good,The agent’s answer states that self‑paced enro...,good,The agent performed one search using a query t...
1,Is a certificate available if I complete the c...,69d122f12e,good,The agent’s answer states that certificates ar...,good,"The agent made a single, relevant search query..."
2,Do self-paced learners get any certificate for...,69d122f12e,good,The agent’s response correctly states that sel...,good,The single search query used relevant keywords...
3,Why are certificates not issued for the self-p...,69d122f12e,good,The agent’s answer correctly states that certi...,good,The agent performed a single search using a qu...
4,Is peer review of capstone projects required i...,69d122f12e,good,The agent’s answer states that a peer review o...,good,The agent issued a single search query that in...


In [31]:
# Calculate the judge cost from the token usage
calc_total_price(usages)

0.013752

In [32]:
# Check the answer scores:
df_agent_eval["answer_score"].value_counts()

,count
answer_score,
good,5


In [33]:
# Check the trajectory scores:
df_agent_eval["trajectory_score"].value_counts()

,count
trajectory_score,
good,5


In [35]:
# Save the judge results:
df_agent_eval.to_csv("agent-evaluations.csv", index=False)